# QuEL-3 Experiment Configure Check

This notebook verifies the end-to-end `Experiment(...); exp.configure(...)` path for QuEL-3.

It shows that the `Experiment` configuration for `QT1` contains both control and readout targets, runs `exp.configure(...)` against a QuEL-3 backend controller configured for server or standalone mode, and then confirms that those targets were deployed through the QuEL-3 configuration manager.

No private monkeypatch is required: the standalone-aware `Quel3BackendController` is injected through the public `Experiment(..., backend_controller=...)` API. Under the current single-active-session assumption, `exp.configure(...)` continues to use the same ambient `SystemManager` controller.


In [ ]:
from pathlib import Path

import qubex as qx
import qubex.system.system_manager as system_manager_module
from qubex.backend.quel3 import Quel3BackendController
from qubex.system.quel3 import Quel3TargetDeployPlanner

In [ ]:
# Example selection
system_id = "144Q-LF-Q3"
if system_id != "144Q-LF-Q3":
    raise ValueError(
        "This notebook is intended for the QuEL-3 example system `144Q-LF-Q3`."
    )

candidate_root_dirs = [
    Path.cwd(),
    Path.cwd() / "docs/examples/system",
    Path.cwd().parent,
    Path.cwd().parent.parent,
]
example_root = next(
    (
        path
        for path in candidate_root_dirs
        if (path / "config").is_dir() and (path / "params").is_dir()
    ),
    None,
)
if example_root is None:
    raise FileNotFoundError(
        "Could not find `docs/examples/system/{config,params}` from the current working directory."
    )

config_dir = example_root / "config"
params_dir = example_root / "params" / system_id

server_host = "localhost"
server_port = 50051
use_standalone_client = True
client_mode = "standalone" if use_standalone_client else "server"
standalone_unit_label = "quel3-02-a01" if use_standalone_client else None

box_ids = ["QT1"]
example_qubits = [f"Q{index:03d}" for index in range(8)]

print(f"example_root: {example_root.resolve()}")
print(f"system_id: {system_id}")
print(f"config_dir: {config_dir.resolve()}")
print(f"params_dir: {params_dir.resolve()}")
print(f"server: {server_host}:{server_port}")
print(f"client_mode: {client_mode}")
print(f"standalone_unit_label: {standalone_unit_label}")
print(f"box_ids: {box_ids}")
print(f"example_qubits: {example_qubits}")

In [ ]:
backend_controller = Quel3BackendController(
    quelware_endpoint=server_host,
    quelware_port=server_port,
    client_mode=client_mode,
    standalone_unit_label=standalone_unit_label,
)

exp = qx.Experiment(
    system_id=system_id,
    qubits=example_qubits,
    config_dir=config_dir,
    params_dir=params_dir,
    backend_controller=backend_controller,
)

print("backend_kind:", exp.system_manager.backend_kind)
print("system_id:", exp.config_loader.system_id)
print("chip_id:", exp.config_loader.chip_id)
print("loaded box_ids:", exp.ctx.box_ids)
print("experiment qubits:", exp.qubit_labels)
print("controller client_mode:", backend_controller.client_mode)

In [ ]:
t1_targets = {
    label: target
    for label, target in sorted(exp.experiment_system.gen_targets.items())
    if target.channel.port.box_id == "QT1"
}
t1_readout_labels = tuple(
    label for label, target in t1_targets.items() if target.type.value == "READ"
)
t1_non_readout_labels = tuple(
    label for label, target in t1_targets.items() if target.type.value != "READ"
)

print("QT1 generator targets:")
for label, target in t1_targets.items():
    print(
        f"- {label}: type={target.type.value}, port={target.channel.port.id}, frequency={target.frequency:.6f} GHz"
    )

print("QT1 readout labels:", t1_readout_labels)
print("QT1 non-readout labels:", t1_non_readout_labels)
print(
    "QT1 has control/readout coverage:",
    len(t1_non_readout_labels) > 0 and len(t1_readout_labels) > 0,
)

In [ ]:
planner = Quel3TargetDeployPlanner()
preview_requests = planner.build_deploy_requests(
    experiment_system=exp.experiment_system,
    box_ids=box_ids,
)
role_counts: dict[str, int] = {}
for request in preview_requests:
    role_counts[request.role] = role_counts.get(request.role, 0) + 1

print("Full deploy requests:")
for request in preview_requests:
    print(
        f"- port_id={request.port_id}, role={request.role}, targets={request.target_labels}"
    )
print("role counts:", role_counts)
print(
    "role coverage check:",
    set(role_counts) == {"TRANSMITTER", "TRANSCEIVER"},
)

In [ ]:
original_confirm_ask = system_manager_module.Confirm.ask
system_manager_module.Confirm.ask = lambda *args, **kwargs: True
try:
    exp.configure(
        box_ids=box_ids,
    )
finally:
    system_manager_module.Confirm.ask = original_confirm_ask

print("exp.configure() completed.")

In [ ]:
target_alias_map = backend_controller.target_alias_map
deployed_infos = backend_controller.last_deployed_instrument_infos
configured_target_labels = tuple(
    request.target_labels[0] for request in preview_requests
)
configured_alias_map = {
    label: target_alias_map.get(label) for label in configured_target_labels
}
missing_target_labels = tuple(
    label for label, alias in configured_alias_map.items() if alias is None
)

print("Configured target -> alias map:")
for label in configured_target_labels:
    print(f"- {label}: {configured_alias_map[label]}")

print("missing target labels:", missing_target_labels)
print("all configured targets deployed:", len(missing_target_labels) == 0)

print("deployed instruments:")
for alias, infos in deployed_infos.items():
    print(f"- {alias}: instrument_count={len(infos)}")

print(
    "full deploy role coverage check:",
    any(label.startswith("Q") for label in configured_target_labels)
    and any(label.startswith("RQ") for label in configured_target_labels),
)